# LLM Feature Extraction — Claude Haiku

Replaces FinBERT binary sentiment labels with rich structured features extracted
by Claude Haiku for each aligned news article. Seven features per article:

- **sentiment_score** — continuous -1.0 to +1.0 (bearish to bullish for WTI)
- **magnitude** — event importance 0.0 to 1.0
- **event_type** — geopolitical / supply / demand / macro / inventory / technical / other
- **entities** — key actors mentioned (OPEC, Russia, Fed, etc.)
- **certainty** — 0.0 speculative to 1.0 confirmed fact
- **price_direction** — bearish / bullish / neutral
- **time_horizon** — immediate / short_term / long_term

Results saved to `llm_features` table in the database.
Resume-capable: re-running skips already-processed articles.

### Imports and setup

In [1]:
import anthropic
import sqlite3
import pandas as pd
import json
import os
import time
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path('../.env').resolve())
client = anthropic.Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))
print('Claude Haiku client ready')

Claude Haiku client ready


### Connect to database and load articles

In [2]:
conn = sqlite3.connect('../01_data/wti_thesis.db')

df_articles = pd.read_sql("""
    SELECT a.id, a.title, a.body, a.body_valid
    FROM articles a
    JOIN liquidity l ON a.id = l.article_id
""", conn)

df_done = pd.read_sql('SELECT article_id FROM llm_features', conn)
done_ids = set(df_done['article_id'].tolist())

df_todo = df_articles[~df_articles['id'].isin(done_ids)].reset_index(drop=True)

print(f'Total aligned articles: {len(df_articles)}')
print(f'Already processed: {len(done_ids)}')
print(f'Remaining to process: {len(df_todo)}')

Total aligned articles: 22795
Already processed: 12024
Remaining to process: 10771


### Feature extraction function

In [39]:
PROMPT_TEMPLATE = """You are a financial analyst specializing in WTI crude oil markets.
Analyze this news article and return ONLY a valid JSON object with no markdown formatting.

{text}

Return exactly this JSON with no backticks, no markdown, just raw JSON:
{{
  \"sentiment_score\": <float -1.0 to 1.0. Positive=bullish for oil, negative=bearish. Focus on NET impact on WTI price>,
  \"magnitude\": <float 0.0 to 1.0. Routine update=0.1, major OPEC cut=0.9, geopolitical crisis=1.0>,
  \"event_type\": <one of: geopolitical, supply, demand, macro, inventory, technical, other>,
  \"entities\": <list of key actors: OPEC, Russia, Fed, Iran, Saudi Arabia, EIA, etc. Empty list if none>,
  \"certainty\": <float 0.0 to 1.0. Rumor=0.1, analyst forecast=0.5, confirmed fact=0.9>,
  \"price_direction\": <one of: bearish, bullish, neutral>,
  \"time_horizon\": <one of: immediate, short_term, long_term. immediate=hours/1day, short_term=days/weeks, long_term=months+>
}}"""

def extract_features(title, body, client):
    # Sanitize inputs
    title = str(title).encode('utf-8', errors='ignore').decode('utf-8').strip()
    body_text = ''
    if isinstance(body, str):
        body_text = body.encode('utf-8', errors='ignore').decode('utf-8')[:800].strip()
    
    text = f'Title: {title}\n\nBody: {body_text}'
    prompt = PROMPT_TEMPLATE.format(text=text)

    message = client.messages.create(
        model='claude-haiku-4-5',
        max_tokens=300,
        messages=[{'role': 'user', 'content': prompt}]
    )

    raw = message.content[0].text.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]
    return json.loads(raw.strip())

print('Feature extraction function ready')

Feature extraction function ready


### Batch processing

Saves to DB every 50 articles. Re-running this cell skips already-processed articles.

In [40]:
SAVE_EVERY = 50
SLEEP_SECONDS = 0.5

results = []
failed_ids = []
total = len(df_todo)

for i, row in df_todo.iterrows():
    try:
        features = extract_features(row['title'], row['body'], client)
        features['article_id'] = int(row['id'])
        features['entities'] = json.dumps(features.get('entities', []))
        results.append(features)

    except Exception as e:
        failed_ids.append(int(row['id']))
        if len(failed_ids) <= 5:
            print(f'Failed article {row["id"]}: {str(e)[:80]}')

    if len(results) > 0 and len(results) % SAVE_EVERY == 0:
        pd.DataFrame(results[-SAVE_EVERY:]).to_sql(
            'llm_features', conn, if_exists='append', index=False
        )
        processed = len(results) + len(done_ids)
        print(f'[{processed}/{total + len(done_ids)}] Saved — failed: {len(failed_ids)}')

    time.sleep(SLEEP_SECONDS)

remaining = len(results) % SAVE_EVERY
if remaining > 0:
    pd.DataFrame(results[-remaining:]).to_sql(
        'llm_features', conn, if_exists='append', index=False
    )

print(f'\n=== Batch complete ===')
print(f'Successfully processed: {len(results)}')
print(f'Failed: {len(failed_ids)}')
if failed_ids:
    print(f'Failed IDs: {failed_ids[:20]}')

[11871/12024] Saved — failed: 0
[11921/12024] Saved — failed: 0
[11971/12024] Saved — failed: 0
[12021/12024] Saved — failed: 0

=== Batch complete ===
Successfully processed: 203
Failed: 0


### Retry failed articles

Run only if there were failures above.

In [41]:
if failed_ids:
    print(f'Retrying {len(failed_ids)} failed articles...')
    retry_results = []
    still_failed = []

    df_failed = df_articles[df_articles['id'].isin(failed_ids)].reset_index(drop=True)

    for _, row in df_failed.iterrows():
        try:
            features = extract_features(row['title'], row['body'], client)
            features['article_id'] = int(row['id'])
            features['entities'] = json.dumps(features.get('entities', []))
            retry_results.append(features)
        except Exception as e:
            still_failed.append(int(row['id']))
        time.sleep(2)

    if retry_results:
        pd.DataFrame(retry_results).to_sql(
            'llm_features', conn, if_exists='append', index=False
        )

    print(f'Recovered: {len(retry_results)}')
    print(f'Still failed: {len(still_failed)}')
else:
    print('No failed articles to retry.')

No failed articles to retry.


### Verify results

In [42]:
total_features = pd.read_sql('SELECT COUNT(*) as n FROM llm_features', conn)['n'][0]
print(f'Total articles with LLM features: {total_features}')

print(f'\nEvent type distribution:')
print(pd.read_sql(
    'SELECT event_type, COUNT(*) as n FROM llm_features GROUP BY event_type ORDER BY n DESC',
    conn
))

print(f'\nPrice direction distribution:')
print(pd.read_sql(
    'SELECT price_direction, COUNT(*) as n FROM llm_features GROUP BY price_direction ORDER BY n DESC',
    conn
))

print(f'\nSentiment score stats:')
print(pd.read_sql(
    'SELECT AVG(sentiment_score) as mean, MIN(sentiment_score) as min, MAX(sentiment_score) as max, AVG(magnitude) as avg_magnitude FROM llm_features',
    conn
))

print(f'\nSample results:')
sample = pd.read_sql("""
    SELECT a.title, f.sentiment_score, f.magnitude, f.event_type, f.price_direction, f.time_horizon
    FROM llm_features f
    JOIN articles a ON f.article_id = a.id
    LIMIT 10
""", conn)
print(sample.to_string())

conn.close()
print('\nDone.')

Total articles with LLM features: 12024

Event type distribution:
     event_type     n
0         macro  3290
1        supply  2280
2         other  2210
3  geopolitical  1656
4        demand  1626
5     technical   571
6     inventory   391

Price direction distribution:
    price_direction     n
0           bearish  5455
1           bullish  4234
2           neutral  2334
3  slightly_bullish     1

Sentiment score stats:
       mean   min   max  avg_magnitude
0 -0.046954 -0.85  0.95       0.371505

Sample results:
                                                                                                 title  sentiment_score  magnitude    event_type price_direction time_horizon
0  Stock market today : Global shares trade higher after Wall Street rally takes S & P 500 near record            -0.15       0.20         macro         bearish   short_term
1                       Goldman Sachs sees crude price rangebound with OPEC+ keeping a lid on capacity             0.30       0.40